# RAG Pipeline - Step by Step Walkthrough

This notebook walks through every stage of the Retrieval-Augmented Generation (RAG) pipeline used in this project:

1. **PDF Loading** - extract raw text from a PDF
2. **Chunking** - split text into overlapping windows
3. **Embedding** - convert chunks to dense vectors
4. **Vector Store** - index vectors with FAISS for fast similarity search
5. **Retrieval** - find top-K relevant chunks for a query
6. **Generation** - feed retrieved context + query to an LLM
7. **Evaluation** - measure retrieval quality


In [ ]:
# Install dependencies (run once)
# !pip install -r ../requirements.txt

## 1. Load the PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = '../data/sample_pdfs/sample.pdf'  # replace with your PDF

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print(f'Pages loaded: {len(documents)}')
print(f'\nSample from page 1:\n{"-"*60}')
print(documents[0].page_content[:500])

## 2. Chunk the Text

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', '.', ' ']
)

chunks = splitter.split_documents(documents)

print(f'Total chunks: {len(chunks)}')
print(f'\nChunk example:\n{"-"*60}')
print(chunks[0].page_content)
print(f'\nMetadata: {chunks[0].metadata}')

### Chunk Size Analysis
Understanding the distribution of chunk lengths.

In [ ]:
import matplotlib.pyplot as plt

lengths = [len(c.page_content) for c in chunks]

plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=30, color='steelblue', edgecolor='white')
plt.title('Chunk Length Distribution')
plt.xlabel('Characters per chunk')
plt.ylabel('Count')
plt.axvline(sum(lengths)/len(lengths), color='red', linestyle='--', label=f'Mean: {sum(lengths)/len(lengths):.0f}')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Embed the Chunks

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Embed a sample chunk to inspect the vector
sample_vector = embeddings.embed_query(chunks[0].page_content)
print(f'Embedding dimension: {len(sample_vector)}')
print(f'Vector sample (first 10 dims): {np.array(sample_vector[:10]).round(4)}')

## 4. Build the FAISS Vector Store

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

print(f'Vector store built with {vectorstore.index.ntotal} vectors')
print(f'Index type: {type(vectorstore.index)}')

## 5. Retrieve Relevant Chunks

In [ ]:
query = 'What is the main topic of the document?'

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4}
)

retrieved_docs = retriever.invoke(query)

print(f'Query: "{query}"')
print(f'\nTop {len(retrieved_docs)} retrieved chunks:\n')
for i, doc in enumerate(retrieved_docs, 1):
    print(f'[{i}] Page {doc.metadata.get("page", "?")}:')
    print(f'    {doc.page_content[:200]}...')
    print()

## 6. Generate an Answer with Qwen2.5

In [ ]:
from operator import itemgetter
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

LLM_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(LLM_MODEL)

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    return_full_text=False,
)
llm = HuggingFacePipeline(pipeline=pipe)

PROMPT_TEMPLATE = """You are a helpful assistant. Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(template=PROMPT_TEMPLATE, input_variables=['context', 'question'])

def format_docs(docs):
    return '\n\n'.join(d.page_content for d in docs)

retrieve = RunnableParallel(
    query=itemgetter('query'),
    source_documents=itemgetter('query') | retriever,
)

answer_chain = (
    {
        'context': RunnableLambda(lambda x: format_docs(x['source_documents'])),
        'question': itemgetter('query'),
    }
    | prompt
    | llm
    | StrOutputParser()
)

chain = retrieve | RunnablePassthrough.assign(result=answer_chain)

result = chain.invoke({'query': query})
print(f'Answer: {result["result"]}')

## 7. Similarity Visualization (Bonus)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Embed all chunks (use first 50 for speed)
sample_chunks = chunks[:50]
vectors = embeddings.embed_documents([c.page_content for c in sample_chunks])
query_vec = embeddings.embed_query(query)

all_vecs = vectors + [query_vec]

pca = PCA(n_components=2)
reduced = pca.fit_transform(all_vecs)

plt.figure(figsize=(10, 6))
plt.scatter(reduced[:-1, 0], reduced[:-1, 1], c='steelblue', alpha=0.6, label='Chunks')
plt.scatter(reduced[-1, 0], reduced[-1, 1], c='red', s=200, zorder=5, label='Query')
plt.title('PCA of Chunk Embeddings vs. Query')
plt.legend()
plt.tight_layout()
plt.show()
print('Red dot = query vector. Closest blue dots = most relevant chunks retrieved.')